In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
processed_dir = Path("../data/processed")

matrice_tfidf = load_npz(processed_dir / "tfidf_matrix_unigrammes.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")

# Il faut s'assurer que la matrice TF-IDF et les identifiants des lignes correspondent
# bien aux produits du DataFrame original.
assert matrice_tfidf.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])

print("Dimensions :", matrice_tfidf.shape)

Dimensions : (500, 2029)


In [3]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=6,
    metric="cosine"
)

labels = dbscan.fit_predict(matrice_tfidf)

# Les résultats restent associés aux identifiants exportés.
clusters = ids.copy()
clusters["cluster"] = labels

display(clusters["cluster"].value_counts().sort_index())

nb_clusters = clusters.loc[
    clusters["cluster"] != -1, "cluster"
].nunique()

print("Nombre de clusters :", nb_clusters)

cluster
-1     333
 0      28
 1      55
 2       6
 3       6
 4       8
 5       6
 6       7
 7      15
 8       6
 9       7
 10     17
 11      6
Name: count, dtype: int64

Nombre de clusters : 12


In [4]:
sans_bruit = labels != -1

X_eval = matrice_tfidf[sans_bruit]
labels_evalues = labels[sans_bruit]

nb_clusters = len(np.unique(labels_evalues))

score = silhouette_score(
    X_eval,
    labels_evalues,
    metric="cosine"
)

print(f"Silhouette hors bruit : {score:.3f}")
print(f"Produits évalués : {sans_bruit.sum()} / {len(labels)}")
print("Bruit (%) :", (labels == -1).mean() * 100)

Silhouette hors bruit : 0.459
Produits évalués : 167 / 500
Bruit (%) : 66.60000000000001


In [5]:
projection_2d = TruncatedSVD(n_components=2, random_state=42)

In [6]:
projection_2d

,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary <random_state>`.",42
,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",2
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [7]:
coordonnees = projection_2d.fit_transform(matrice_tfidf)
visualisation = pd.DataFrame({
    "SVD 1": coordonnees[:, 0],
    "SVD 2": coordonnees[:, 1],
    "id": ids["id"].to_numpy(),
    "cluster": [
        "Bruit" if label == -1 else f"Cluster {label}"
        for label in labels
    ]
})

In [8]:
coeurs = np.zeros(len(labels), dtype=bool)
coeurs[dbscan.core_sample_indices_] = True

visualisation["statut"] = np.where(
    labels == -1,
    "Bruit",
    np.where(coeurs, "Cœur", "Bordure")
)

fig = px.scatter(
    visualisation,
    x="SVD 1",
    y="SVD 2",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — clusters et vrais points cœur",
    height=600,
    width=600
)

fig.show()

In [9]:
tsne = TSNE(
    n_components=2,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_tsne = tsne.fit_transform(matrice_tfidf.toarray())

visu_tsne = visualisation.copy()
visu_tsne["t-SNE 1"] = coord_tsne[:, 0]
visu_tsne["t-SNE 2"] = coord_tsne[:, 1]

fig = px.scatter(
    visu_tsne,
    x="t-SNE 1",
    y="t-SNE 2",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — visualisation t-SNE, distance cosinus",
    height=600,
    width=600
)

fig.show()

In [10]:
tsne_3d = TSNE(
    n_components=3,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_3d = tsne_3d.fit_transform(matrice_tfidf.toarray())

visu_3d = visualisation.copy()
visu_3d[["t-SNE 1", "t-SNE 2", "t-SNE 3"]] = coord_3d

fig = px.scatter_3d(
    visu_3d,
    x="t-SNE 1",
    y="t-SNE 2",
    z="t-SNE 3",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — projection t-SNE en 3D",
    height=600,
    width=600
)

fig.update_traces(marker_size=4)
fig.update_layout(scene_aspectmode="data")
fig.show()

In [12]:
vectorizer = joblib.load("../models/tfidf_vectorizer_unigrammes.joblib")
mots = vectorizer.get_feature_names_out()

# Retrouver les descriptions dans l'ordre exact de la matrice
source = pd.read_csv(
    processed_dir / "sample_clean.csv"
).set_index("id")

lecture = ids.copy()
lecture["cluster"] = labels
lecture["texte"] = source.loc[
    ids["id"], "description_texte"
].to_numpy()

assert matrice_tfidf.shape == (len(lecture), len(mots))

In [13]:
cluster_ids = np.unique(labels[labels != -1]) # Parcours tous les clusters sauf le bruit (-1)

for cluster_id in cluster_ids:
    masque = labels == cluster_id
    assert cluster_id != -1 and masque.any()

    X_cluster = matrice_tfidf[masque]

    # Poids TF-IDF moyen de chaque mot dans ce groupe
    poids = np.asarray(X_cluster.mean(axis=0)).ravel()
    top = np.argsort(poids)[::-1][:12]
    top = top[poids[top] > 0]

    mots_cluster = pd.DataFrame({
        "mot": mots[top],
        "poids TF-IDF moyen": poids[top]
    })

    fig = px.bar(
        mots_cluster,
        x="poids TF-IDF moyen",
        y="mot",
        orientation="h",
        title=f"Cluster {cluster_id} — mots dominants",
        height=400,
        width=600
    )
    fig.update_yaxes(categoryorder="total ascending")
    fig.show()

    # Descriptions les plus proches du centre moyen en distance cosinus
    centre = np.asarray(X_cluster.mean(axis=0))
    similarites = cosine_similarity(X_cluster, centre).ravel()

    positions = np.flatnonzero(masque)
    selection = positions[np.argsort(similarites)[::-1][:3]]

    print(f"Cluster {cluster_id} : {len(positions)} produits")

    with pd.option_context("display.max_colwidth", None):
        display(lecture.iloc[selection][["id", "texte"]])

Cluster 0 : 28 produits


,id,texte
496,497,"cap 2 crew - crew takes edge fickle weather . clearing conditions , fastest dry time performance baselayer fabrics , open - knit construction breathes wicks dry cool . weather window shuts , wear insulation beneath layers . self - fabric neck s soft skin , raglan sleeves flat - ribbed underarm panels merge way shoulder straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
21,22,"cap 2 t - shirt - revels sweat jamborees hill climbs , ridge runs desert singletrack resolute sun . fastest - drying performance baselayer fabrics , open knit invites airflow provides excellent wicking exceptional breathability . worn beneath layers , s insulative chilly days . self - binding neck pliable soft raglan sleeves flat - ribbed underarm panels merge way straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
358,359,"cap 2 cap sleeve - fastest - drying performance baselayer fabrics , capilene 2 open knit invites airflow cool provides excellent wicking exceptional breathability dry . s insulative chilly days winter hasn t released grip . minimalist tech features feminine v - neck self - fabric neckline , easy - moving underarm panels provide shaping . raglan - style sleeves stay way pack straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."


Cluster 1 : 55 produits


,id,texte
115,116,"rockpile t - shirt - beauty bouldering stacked rockpile t - shirt , featuring original artwork elissa pfost . printed 100 % ringspun organic cotton pvc- phthalate - free inks , shirt taped shoulder seams won t chafe pack . recyclable common threads recycling program ."
123,124,"squid t - shirt - artist tim tompkinson s original work renders squid t - shirt coat arms . printed ringspun organic cotton 100 % , lightweight shirt printed pvc- phthalate - free inks harmful environment . taped shoulder seams provide - chafe fit . recyclable common threads recycling program ."
137,138,"text logo t - shirt - text logo t - shirt tells classic story word . printed pure , 100 % organic cotton pvc- phthalate - free inks , shirt ringspun softness . durable , taped shoulder seams minimize chafing . recyclable common threads recycling program ."


Cluster 2 : 6 produits


,id,texte
30,31,"crosstown - crosstown cross - country , fully featured commuter backpack ensures life stays organized office - bound , hiking steep narrow , global commute . inside zippered main compartment , features suspended , self - adjusting padded laptop sleeve handles 15 17 laptop . secondary zippered compartment opens portable office organizer system . catch - sleeve hidden zipper provides easy access documents stretch - woven water - bottle pocket fleece - lined pocket protects sunglasses electronics 3 - d spacer - mesh backpack straps padding comfort breathability adjustable sternum strap built - safety whistle . 150 - denier - recycled polyester double weave polyurethane coating dwr durable water repellent finish ."
463,464,"daily grind - fully padded shoulder bag keeps moving unchallenged daily grind . minimalist design , protects 17 smaller laptop helps stay organized . main pocket stretch - woven internal sleeve accessed simple flap hook - - loop closure . organizer pockets zippered security pocket smaller gear fingertips , easy - access document sleeve holds airline tickets , reading material notebook . carry long , adjustable shoulder strap - mounted handle . 150 - denier - recycled polyester double weave polyurethane coating dwr durable water repellent finish ."
435,436,"lightwire - need leave office rearview mirror . lightwire plenty room jacket lunch addition fully padded computer sleeve , protects 15 17 pc laptop . dedicated flat storage section holds files magazines , external pocket offers - handed access office organizer . highly breathable 3 - d spacer - mesh shoulder straps padded panel wick away moisture . features include mp3 pocket , sternum strap built - safety whistle , tuck - away waistbelt , stretch water - bottle pocket easy - access external security pocket . 150 - denier - recycled polyester double weave polyurethane coating dwr durable water repellent finish ."


... (14 outputs left out)

Cluster 10 : 17 produits


,id,texte
381,382,"rain shadow jkt - staying inside isn t way stay dry - enjoy trails conditions fully seam - sealed waterproof / breathable protection . rain shadow packable , lightweight tear - resistant 2.5 - layer 100 % nylon fabric waterproof / breathable h2no barrier deluge dwr durable water repellent finish . inside , raised pattern diffuses channels moisture away skin , aids hassle - free layering , enhances long - term durability . updated microfleece lining neck chin , waterproof handwarmer pockets longer mesh - lined zipper garages , external zippers water - resistant storm flaps . features roll - , 2 - way - adjustable , helmet - compatible hood , 2 - way center - zipper , self - fabric cuff tabs drawcord hem ."
211,212,"rain shadow jkt - staying inside isn t way stay dry - enjoy trails conditions fully seam - sealed waterproof / breathable protection . rain shadow packable , lightweight tear - resistant 2.5 - layer 100 % nylon fabric waterproof / breathable h2no barrier deluge dwr durable water repellent finish . inside , raised pattern diffuses channels moisture away skin , aids hassle - free layering , enhances long - term durability . updated microfleece lining neck chin , waterproof handwarmer pockets longer mesh - lined zipper garages , external zippers water - resistant storm flaps . features roll - , 2 - way - adjustable , helmet - compatible hood , 2 - way center - zipper , self - fabric cuff tabs drawcord hem ."
273,274,"torrentshell jkt - tropical rain forests share thing patagonia ice cap wet weather . trekking hiking drenching conditions , new torrentshell provides 2.5 - layer waterproof / breathable protection s durable , lightweight frills . jacket stuffs zippered handwarmer pockets clip - loop hood fits snugly helmet , rolls , stows away , laminated visor center zipper exterior interior storm flaps . pit zips added ventilation , microfleece - lined neck chin drawcord hem ."


Cluster 11 : 6 produits


,id,texte
483,484,"borderless trek pants - ruby crest trail runs directions . moving shifting terrain , borderless trek pants stretchy blend 95 % nylon 60 % recycled 5 % spandex , woven soft , flexible fabric s equal parts tough comfortable offers 40 - upf sun protection . waist includes partially elasticized band motion - friendly fit gusseted crotch ensures dexterity stream crossings talus hopping . pockets round generous pocket brigade . inseam size 32 32 . recyclable common threads recycling program ."
484,485,"borderless trek zip - pants - technical performance wide spectrum mountain temperatures , versatile pants zip - legs quickly - coverage pants shorts match prevailing conditions . 95 % nylon 60 % recycled 5 % spandex fabric lends durability rough terrain , excellent airflow , generous stretch mobility 40 - upf sun protection . motion - inspired features include gusseted crotch partially elasticized waistband . pockets , , thigh hidden zip - security pocket . pants inseam size 32 32 shorts inseam 12 1/2 . recyclable common threads recycling program ."
14,15,"borderless shorts-11 . - forward cringe . borderless shorts prying outer reaches unnamed terrain . light , breathable fabric blends 95 % nylon 60 % recycled 5 % spandex mini ripstop resist effects thrashy chimneys provide stretch body language going places . dwr durable water repellent finish shrugs water built - 40 - upf sun protection shades legs open country . features include gusseted crotch preserve mobility , pockets access wearing harness partially elastic waist shorts adhered load hip belt . rear welted pockets secure zips . inseam size m 11 . recyclable common threads recycling program ."
